# TP1 — Téléchargement automatique des PDF du corpus CAMille

Ce notebook récupère les liens PDF présents sur la page CAMille, puis télécharge automatiquement les 51 fichiers demandés.

## Imports

In [1]:
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

## Récupération des liens PDF de CAMille


In [2]:
page_url = "https://max.de.wilde.web.ulb.be/camille/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(page_url, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

pdf_links = sorted({
    urljoin(page_url, link["href"])
    for link in soup.find_all("a", href=True)
    if urlparse(link["href"]).path.lower().endswith(".pdf")
})

print(f"Nombre de liens PDF trouvés : {len(pdf_links)}")
pdf_links[:5]

Nombre de liens PDF trouvés : 51


['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB230_1913-07-05_01-0001.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB258_1884-09-03_01-0003.pdf',
 'https://max.de.wilde.web.ulb.be/camille/KB_JB258_1894-12-09_01-0003.pdf']

In [3]:
assert len(pdf_links) == 51, (
    f"Le script devait trouver 51 PDF, mais en a trouvé {len(pdf_links)}."
)

print("Vérification réussie : les 51 liens PDF ont été trouvés.")

Vérification réussie : les 51 liens PDF ont été trouvés.


In [4]:
output_dir = Path("pdf_camille")
output_dir.mkdir(parents=True, exist_ok=True)

downloaded_files = []
failed_downloads = []

for index, pdf_url in enumerate(pdf_links, start=1):
    filename = Path(urlparse(pdf_url).path).name
    destination = output_dir / filename

    try:
        pdf_response = requests.get(
            pdf_url,
            headers=headers,
            timeout=60
        )
        pdf_response.raise_for_status()

        if not pdf_response.content.startswith(b"%PDF"):
            raise ValueError("Le fichier reçu n'est pas un PDF valide.")

        destination.write_bytes(pdf_response.content)
        downloaded_files.append(destination)
        print(f"[{index:02d}/51] Téléchargé : {filename}")

    except (requests.RequestException, ValueError) as error:
        failed_downloads.append((pdf_url, str(error)))
        print(f"[{index:02d}/51] Échec : {filename} — {error}")

[01/51] Téléchargé : KB_JB230_1892-08-07_01-0003.pdf
[02/51] Téléchargé : KB_JB230_1903-10-16_01-0002.pdf
[03/51] Téléchargé : KB_JB230_1913-07-05_01-0001.pdf
[04/51] Téléchargé : KB_JB258_1884-09-03_01-0003.pdf
[05/51] Téléchargé : KB_JB258_1894-12-09_01-0003.pdf
[06/51] Téléchargé : KB_JB258_1906-01-09_01-0002.pdf
[07/51] Téléchargé : KB_JB421_1899-05-15_01-00003.pdf
[08/51] Téléchargé : KB_JB421_1926-10-29_01-00002.pdf
[09/51] Téléchargé : KB_JB421_1950-04-15_01-00004.pdf
[10/51] Téléchargé : KB_JB427_1920-01-10_01-00004.pdf
[11/51] Téléchargé : KB_JB427_1933-01-04_01-00003.pdf
[12/51] Téléchargé : KB_JB427_1949-07-18_01-00008.pdf
[13/51] Téléchargé : KB_JB449_1846-05-30_01-00002.pdf
[14/51] Téléchargé : KB_JB449_1912-01-04_01-00003.pdf
[15/51] Téléchargé : KB_JB449_1947-08-29_01-00003.pdf
[16/51] Téléchargé : KB_JB494_1853-10-30_01-0002.pdf
[17/51] Téléchargé : KB_JB494_1922-09-28_01-0005.pdf
[18/51] Téléchargé : KB_JB494_1939-12-08_01-0004.pdf
[19/51] Téléchargé : KB_JB555_1836-02

In [5]:
saved_pdfs = sorted(output_dir.glob("*.pdf"))

print(f"PDF enregistrés : {len(saved_pdfs)}")
print(f"Échecs : {len(failed_downloads)}")
print(f"Dossier : {output_dir.resolve()}")

assert len(saved_pdfs) == 51
assert not failed_downloads

print("Téléchargement terminé et vérifié avec succès.")

PDF enregistrés : 51
Échecs : 0
Dossier : C:\Users\HP\Documents\tac\tps\tp1\pdf_camille
Téléchargement terminé et vérifié avec succès.


Cinq documents PDF datés de 1836, 1894, 1920, 1939 et 1950 ont été examinés afin de comparer la langue, les thèmes, la mise en page et la qualité de l’OCR, et d’obtenir un premier aperçu de l’évolution et de la diversité du corpus.